***
# Process NOAA weather observations

This notebook reduces a raw NOAA Local Climatological Data export to the hourly weather fields needed by the capstone workflow. It processes one station and calendar year at a time, removes columns and records that carry no usable observations, and writes the result to the processed-data directory.

> **Scope:** `STATION` identifies the NOAA station used as the weather source, while `AIRPORT` controls the output filename. Keep this mapping consistent when changing either value.
***

In [57]:
YEAR = 2024

# STATION = 74486094789
# AIRPORT = "JFK"
# STATION = 72530094846
# AIRPORT = "ORD"
STATION = 72219013874
AIRPORT = "ATL"

RAW_DATA_FILE = f"../data/noaa/raw/{YEAR}/{STATION}.csv"
PROCESSED_DATA_FILE = f"../data/noaa/processed/{AIRPORT}_{YEAR}.csv"

***
## Load the raw NOAA export

`low_memory=False` asks pandas to infer each column's dtype using the complete file. NOAA weather fields can contain measurement flags and trace-value markers, so some apparently numeric columns may still load as strings and should be normalized later before modeling.
***

In [58]:
import pandas as pd

df = pd.read_csv(RAW_DATA_FILE, low_memory=False)

***
## Remove columns with no observations

Columns that are missing for every row contain no information for this station-year. The list is printed before removal so changes in NOAA coverage remain visible when the notebook is rerun for another station or year.
***

In [59]:
null_only_columns = df.columns[df.isna().all()].tolist()

print(f"Columns containing only null/NaN values: {len(null_only_columns)}")
print(null_only_columns)

df = df.drop(columns=null_only_columns)
df.shape

Columns containing only null/NaN values: 13
['MonthlyAverageRH', 'MonthlyDewpointTemperature', 'MonthlyGreatestSnowDepthDate', 'MonthlyWetBulb', 'BackupDirection', 'BackupDistance', 'BackupDistanceUnit', 'BackupElements', 'BackupElevation', 'BackupEquipment', 'BackupLatitude', 'BackupLongitude', 'BackupName']


(13446, 112)

***
## Retain the capstone's weather feature set

The following exclusion list removes station metadata, reporting metadata, aggregate daily/monthly fields, backup fields, and weather measurements not used downstream. Keeping hourly observations separate from daily and monthly summaries avoids mixing different temporal granularities in the modeling table.

`errors="ignore"` makes the drop operation reusable across NOAA files whose schemas differ slightly. It also means a misspelled column name will not raise an error, so this list should be reviewed whenever the upstream schema or intended model features change.
***

In [60]:
columns_to_drop = [
    "STATION",
    "LATITUDE",
    "LONGITUDE",
    "NAME",
    "ELEVATION",
    "REPORT_TYPE",
    "SOURCE",
    "HourlyAltimeterSetting",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlySkyConditions",
    "HourlyWetBulbTemperature",
    "HourlyWindGustSpeed",
    "Sunrise",
    "Sunset",
    "DailyAverageDewPointTemperature",
    "DailyAverageDryBulbTemperature",
    "DailyAverageRelativeHumidity",
    "DailyAverageSeaLevelPressure",
    "DailyAverageStationPressure",
    "DailyAverageWetBulbTemperature",
    "DailyAverageWindSpeed",
    "DailyCoolingDegreeDays",
    "DailyDepartureFromNormalAverageTemperature",
    "DailyHeatingDegreeDays",
    "DailyMaximumDryBulbTemperature",
    "DailyMinimumDryBulbTemperature",
    "DailyPeakWindDirection",
    "DailyPeakWindSpeed",
    "DailyPrecipitation",
    "DailySnowDepth",
    "DailySnowfall",
    "DailySustainedWindDirection",
    "DailySustainedWindSpeed",
    "DailyWeather",
    "MonthlyAverageRH",
    "MonthlyDaysWithGT001Precip",
    "MonthlyDaysWithGT010Precip",
    "MonthlyDaysWithGT32Temp",
    "MonthlyDaysWithGT90Temp",
    "MonthlyDaysWithLT0Temp",
    "MonthlyDaysWithLT32Temp",
    "MonthlyDepartureFromNormalAverageTemperature",
    "MonthlyDepartureFromNormalCoolingDegreeDays",
    "MonthlyDepartureFromNormalHeatingDegreeDays",
    "MonthlyDepartureFromNormalMaximumTemperature",
    "MonthlyDepartureFromNormalMinimumTemperature",
    "MonthlyDepartureFromNormalPrecipitation",
    "MonthlyDewpointTemperature",
    "MonthlyGreatestPrecip",
    "MonthlyGreatestPrecipDate",
    "MonthlyGreatestSnowDepth",
    "MonthlyGreatestSnowDepthDate",
    "MonthlyGreatestSnowfall",
    "MonthlyGreatestSnowfallDate",
    "MonthlyMaxSeaLevelPressureValue",
    "MonthlyMaxSeaLevelPressureValueDate",
    "MonthlyMaxSeaLevelPressureValueTime",
    "MonthlyMaximumTemperature",
    "MonthlyMeanTemperature",
    "MonthlyMinSeaLevelPressureValue",
    "MonthlyMinSeaLevelPressureValueDate",
    "MonthlyMinSeaLevelPressureValueTime",
    "MonthlyMinimumTemperature",
    "MonthlySeaLevelPressure",
    "MonthlyStationPressure",
    "MonthlyTotalLiquidPrecipitation",
    "MonthlyTotalSnowfall",
    "MonthlyWetBulb",
    "AWND",
    "CDSD",
    "CLDD",
    "DSNW",
    "HDSD",
    "HTDD",
    "DYTS",
    "DYHF",
    "NormalsCoolingDegreeDay",
    "NormalsHeatingDegreeDay",
    "ShortDurationEndDate005",
    "ShortDurationEndDate010",
    "ShortDurationEndDate015",
    "ShortDurationEndDate020",
    "ShortDurationEndDate030",
    "ShortDurationEndDate045",
    "ShortDurationEndDate060",
    "ShortDurationEndDate080",
    "ShortDurationEndDate100",
    "ShortDurationEndDate120",
    "ShortDurationEndDate150",
    "ShortDurationEndDate180",
    "ShortDurationPrecipitationValue005",
    "ShortDurationPrecipitationValue010",
    "ShortDurationPrecipitationValue015",
    "ShortDurationPrecipitationValue020",
    "ShortDurationPrecipitationValue030",
    "ShortDurationPrecipitationValue045",
    "ShortDurationPrecipitationValue060",
    "ShortDurationPrecipitationValue080",
    "ShortDurationPrecipitationValue100",
    "ShortDurationPrecipitationValue120",
    "ShortDurationPrecipitationValue150",
    "ShortDurationPrecipitationValue180",
    "REM","BackupDirection",
    "BackupDistance",
    "BackupDistanceUnit",
    "BackupElements",
    "BackupElevation",
    "BackupEquipment",
    "BackupLatitude",
    "BackupLongitude",
    "BackupName",
    "WindEquipmentChangeDate"
]

In [61]:
df = df.drop(columns=columns_to_drop, errors="ignore")

***
## Remove records with no retained weather measurements

A row is removed only when **every retained feature except `DATE` is missing**. Rows with partial weather coverage are preserved for later imputation or feature-specific filtering; this is not the same as dropping every row that contains any missing value.

> `DATE` remains the observation key. Convert it with `pd.to_datetime` and choose a consistent timezone in the downstream feature pipeline before extracting calendar or elapsed-time features.
***

In [62]:
feature_columns = df.columns.drop("DATE") # DATE is never NaN
all_feature_nan_rows = df[feature_columns].isna().all(axis=1)

print(f"Rows with all non-DATE features NaN: {all_feature_nan_rows.sum()}")

df = df.loc[~all_feature_nan_rows].reset_index(drop=True)
df.shape

Rows with all non-DATE features NaN: 376


(13070, 9)

***
## Write the processed station-year file

The parent directory is created when necessary, then the cleaned dataframe is saved without a pandas index. The CSV contains the original `DATE` values plus the retained NOAA weather fields; numeric coercion, missing-value treatment, and model feature engineering should be applied consistently downstream.
***

In [63]:
from pathlib import Path

Path(PROCESSED_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_FILE, index=False)